# Classify SMM diagnosis in notes

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import json
import logging

import pandas as pd

from smmnlp.config import ModelConfig
from smmnlp.inference import generate_responses
from smmnlp.modeling import LocalLLM
from smmnlp.paths import ProjectPaths
from smmnlp.prompts import Task

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

In [ ]:
generator = LocalLLM.from_pretrained(ModelConfig(paths.model_path))

In [ ]:
readfile = paths.data("Update/ExternalValidation/external_sample100_notes.json")
with open(readfile, "r", encoding="utf-8", errors="ignore") as f:
    data = json.load(f)

In [ ]:
notes = pd.DataFrame(data)

In [ ]:
note_count = len(notes)

In [ ]:
print(note_count)

In [ ]:
notes.head(5)

In [ ]:
responses = generate_responses(notes["full_note"].tolist(), generator, Task.DIAGNOSIS)

In [ ]:
notes["SMMDx_LLM_Binary"] = responses

In [ ]:
notes.head(10)

In [ ]:
savefile_name = paths.data("Update/ExternalValidation/Results/phase1_fewshot_binary.xlsx")
notes.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)